In [ ]:
import pandas as pd

# Se carga el archivo para poder leer "manipular"
data_frame = pd.read_csv('server_logs.csv')
#print(data_frame.to_string())   ->  Print de verificacion

# Convertir el timestamp_event en objetos tipo datetime
data_frame['timestamp_event'] = pd.to_datetime(data_frame['timestamp_event'])
# print(data_frame['timestamp_event'])  -> Print de verificacion

# Creamos una columna para verificar "Bad Event"
data_frame['malo'] = (data_frame['severity'].isin(['ERROR', 'CRITICAL'])) | (data_frame["status_code"] >= 500)
#print(data_frame['malo'])   ->  Print de verificacion

#Definiciones Operativas - Time Window(bin) - Agrupacion en ventanas de 5 minutos
data_frame_resumen = data_frame.set_index('timestamp_event').resample('5min').agg(
    total_events = ('service_name', 'count'),
    bad_events = ('malo', 'sum'),
    average_latency = ('latency_ms', 'mean')     # Averiguar bien en que etapa y para que se usa
)

# Calculo del Bad Rate
data_frame_resumen['badRate'] = data_frame_resumen['bad_events'] / data_frame_resumen['total_events']
# print(data_frame_resumen['Bad Rate'])   -> Print de verificacion

# Deteccion momento critico, total_events (minimo 20), ordenamo por el peor Bad Rate
peores_ventanas = data_frame_resumen[data_frame_resumen['total_events'] >= 20].sort_values('badRate', ascending=False)
inicio_momento_critico = peores_ventanas.index[0]

# Diagnostico de lo ocurrido en los 5min
# Se filtra el data_frame original, solo para ese momento
data_inicident =  data_frame[(data_frame['timestamp_event'] >= inicio_momento_critico) &
                            (data_frame['timestamp_event'] < inicio_momento_critico + pd.Timedelta(minute=5))]

# Comparacion del incidente vs Baseline(baseline es todo lo que NO es el momento critico, incidente)
data_baseline = data_frame[data_frame['timestamp_event'] != inicio_momento_critico] 


Total de logs

In [ ]:

total_logs = data_frame.shape[0]
#print("Cantidad total de logs:", total_logs)     -> print de verificacion

Cantidad total de logs: 5795


Por severidad

In [ ]:
#data_frame['severity'].unique()
#data_frame.groupby('severity').size()

tiposDeSeveridad = data_frame['severity'].value_counts()
severidadMasComun = data_frame['severity'].value_counts().idxmax()
#print(tiposDeSeveridad, severidadMasComun)     -> Print de verificacion


Los tipo de severidad son: severity
INFO        3542
WARN        1358
ERROR        775
CRITICAL     120
Name: count, dtype: int64 y la severidad mas comun es: INFO


Servicio con mas logs

In [ ]:
# Conteo de servicios y cantidad de apariciones
conteoLogs = data_frame['service_name'].value_counts()
servicioMasLogs = data_frame['service_name'].value_counts().idxmax()
#print(conteoLogs, servicioMasLogs)         -> Print de verificacion

service_name
api-gateway             1509
orders-service          1057
inventory-service        964
payment-service          842
auth-service             778
notification-service     645
Name: count, dtype: int64 api-gateway


Servicio con menos logs

In [ ]:
servicioMenosLogs = data_frame['service_name'].value_counts().idxmin()  
#print(servicioMenosLogs)      -> Print de verificacion

notification-service
